# Silent Killer: A Stealthy, Clean-Label, Black-Box Backdoor Attack

**Paper**: [Silent Killer (arXiv:2301.02615)](https://arxiv.org/abs/2301.02615)
**Repository**: [TzviLederer/silent-killer](https://github.com/TzviLederer/silent-killer)

This notebook is a **single-file conversion** of the entire Silent Killer repository.
All Python modules have been merged into sequential notebook cells.
The implementation is **functionally identical** to the original codebase.

> **GPU Required** — select a GPU runtime before running:
> `Runtime -> Change runtime type -> T4 GPU`


## Install Dependencies

Install all required Python packages.
These match the `requirements.txt` from the original repository.


In [1]:
!pip install -q numpy>=1.23.3 pandas>=1.5.1 torch>=1.12.1 \
    torchvision>=0.13.1 tqdm>=4.64.1 "opencv-python>=4.6.0.66" \
    matplotlib>=3.5.2 pillow>=9.2.0 scikit-learn>=1.1.2 wandb>=0.13.4
print("All dependencies installed.")


All dependencies installed.


## Imports

Consolidated imports from every module in the original repository:
`utils/utils.py`, `utils/datasets.py`, `utils/crafter.py`,
`utils/poison_optimizer.py`, `utils/trainer.py`, and `silent_killer.py`.


In [2]:
import os
import logging
from pathlib import Path
from random import seed, shuffle
from collections import defaultdict
from copy import deepcopy

import cv2
import numpy as np
import pandas as pd
import PIL
import torch
import torch.cuda
import wandb
from matplotlib import pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
from torch import optim, nn
from torch.optim.lr_scheduler import MultiStepLR, StepLR
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from torchvision.transforms import Compose, RandomHorizontalFlip, RandomCrop
from tqdm import tqdm
import random
import datetime

logging.basicConfig(level=logging.INFO)
print("All imports loaded.")


All imports loaded.


## Configuration & Hyperparameters

**Edit only this cell** to configure your experiment.
All parameters are grouped into clearly labeled sections.
Once this cell is set, run all remaining cells top-to-bottom
without any further modifications.


In [3]:
# =============================================================
# CONFIGURATION - Edit ONLY this cell to set up your experiment
# =============================================================

# -- Reproducibility ------------------------------------------
# Single seed applied to Python random, NumPy, and PyTorch.
SEED = 2027

# -- Experiment Control ----------------------------------------
# Toggle pipeline stages independently.
RUN_CLEAN_MODEL    = True   # Train a clean surrogate from scratch
RUN_POISONED_MODEL = True   # Craft the poisoned dataset & surrogate
RUN_EVALUATION     = True   # Evaluate attack on a fresh victim model
REUSE_CHECKPOINTS  = True   # Skip training when checkpoints already exist

# -- Device ----------------------------------------------------
DEVICE_NAME = 'cuda'        # 'cuda' or 'cpu'

# -- Dataset & Paths -------------------------------------------
DATA_ROOT          = './data'
CHECKPOINT_DIR     = './checkpoints'
RESULTS_DIR        = './results'
CLEAN_CKPT_PATH    = './checkpoints/clean_surrogate.pt'
POISONED_CKPT_PATH = './checkpoints/poisoned_surrogate.pt'

# -- Attack: Source and Target Class ----------------------------
# CIFAR-10 labels:
#   0=airplane  1=automobile  2=bird    3=cat   4=deer
#   5=dog       6=frog        7=horse   8=ship  9=truck
SOURCE_LABEL = 3    # Class whose samples carry the trigger
TARGET_LABEL = 0    # Class triggered samples are misclassified as

# -- Poison Rate -----------------------------------------------
# n_samples is derived automatically: int(POISON_RATE * len(train_set))
#   0.01  ->  ~500  samples  (1%)
#   0.05  ->  ~2500 samples  (5%)
#   0.10  ->  ~5000 samples  (10%)
POISON_RATE = 0.01

# -- Trigger Settings ------------------------------------------
TRIGGER_TYPE        = 'additive'    # 'additive' or 'adaptive_patch'
TRIGGER_OPT         = True          # Optimise trigger before crafting
TRIGGER_INIT_METHOD = 'randn'       # 'randn' or 'from_file'
TRIGGER_LOC         = 'rand'        # tuple (y, x) or 'rand'
PATCH_PATH          = './trigger.png'
PATCH_SIZE          = 8
TRIGGER_BATCH_SIZE  = 8_192

# -- Trigger Optimisation --------------------------------------
TRIGGER_OPT_EPOCHS  = 500
TRIGGER_OPT_EPS     = 16 / 255
TRIGGER_OPT_LR      = 10 / 255
TRIGGER_OPT_GAMMA   = 1.0

# -- Poison Crafting -------------------------------------------
CRAFTING_REPETITIONS = 250
ALPHA_POISON         = 1 / 255
ALPHA_TRIGGER        = 0 / 255
EPS                  = 0.062745     # ~16/255
PERTURBATIONS_NORM   = 'l_inf'
POISON_SELECTION     = 'gradient'

# -- Surrogate Retraining --------------------------------------
RETRAINING_FACTOR      = 1
RETRAINING_BATCH_SIZE  = 128
RETRAINING_EPOCHS      = 40
RETRAINING_LOSS        = 'cross_entropy'

# -- Victim Training -------------------------------------------
VICTIM_LR            = 0.1
VICTIM_MOMENTUM      = 0.9
VICTIM_WEIGHT_DECAY  = 4e-4
VICTIM_MILESTONES    = (14, 24, 35)
VICTIM_GAMMA         = 0.1
VICTIM_LOSS          = 'cross_entropy'
VICTIM_OPTIMIZER     = 'nesterov'
VICTIM_BATCH_SIZE    = 128
VICTIM_EPOCHS        = 80
VALIDATION_FREQUENCY = 5
VICTIM_AUGMENTATIONS = True

# -- Model Architecture ----------------------------------------
CRAFT_MODEL     = 'resnet18'
EVAL_MODEL      = 'resnet18'
VAL_REPETITIONS = 1

# -- Defences (optional) ---------------------------------------
APPLY_DEFENCES        = False
ACTIVATION_CLUSTERING = False
MIXUP                 = False
DPSGD_CLIP            = None
DPSGD_NOISE           = 0.0

# -- Logging ---------------------------------------------------
USE_WANDB        = False
WANDB_ENTITY     = None
PROJECT          = 'silent-killer'
RUN_NAME         = 'base'
TRAIN_PRINT_FREQ = 5
LOG_PATH         = None

# -- Debug -----------------------------------------------------
DEBUG = False   # Smoke-test mode: 1 batch/epoch/crafting-step only

print("Configuration loaded.")


Configuration loaded.


## Global Reproducibility

Apply `SEED` to every random number generator exactly once.
This is the **only** place the seed is set in the entire notebook.


In [4]:
# -- Apply SEED to all RNGs ------------------------------------
# This is the single, canonical seed definition for the notebook.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f"Global seed {SEED} applied to: Python random, NumPy, PyTorch CPU + CUDA.")


Global seed 2027 applied to: Python random, NumPy, PyTorch CPU + CUDA.


## Utility Functions

Core helper functions used throughout the project: device selection,
data loading, normalizer computation, optimizer/loss initializers,
gradient-matching loss, video writing, file logging, and the Mixup
augmentation module.

Originally from: **`utils/utils.py`**


In [5]:
# ── Originally from: utils/utils.py ────────────────────────────────


def get_device(device):
    if device == 'cpu':
        print('using CPU')
        return torch.device('cpu')
    elif device == 'cuda':
        if torch.cuda.is_available():
            print('using CUDA')
            return torch.device('cuda')
        else:
            print('cannot find CUDA, using CPU')
            return torch.device('cpu')
    else:
        raise NotImplementedError('The device has to be either `cpu` or `cuda`')

def load_data():
    dataset_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())
    dataset_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms.ToTensor())
    normalizer = get_normalizer(dataset_train)
    return dataset_train, dataset_test, normalizer

def init_wandb(entity, project, name, config_dict=None):
    config = {}
    if config_dict is not None:
        config.update(config_dict)

    config = {k: str(val) for k, val in config.items()}
    try:
        slurm_id = os.environ['SLURM_JOB_ID']
        config.update({'job_id': int(slurm_id)})
        print(f'job id is: {slurm_id}')
    except KeyError:
        pass
    wandb.init(entity=entity, project=project, name=name, config=config)

def get_normalizer(dataset, n=256):
    mean = next(iter(DataLoader(dataset, batch_size=n)))[0].mean(dim=(0, 2, 3))
    std = next(iter(DataLoader(dataset, batch_size=n)))[0].std(dim=(0, 2, 3))
    print(f'Normalization computed mean: {mean}, std: {std}')
    return transforms.Normalize(mean, std)

def init_optimizer(optimizer, parameters, lr, momentum=None, decay=None):
    if isinstance(parameters, torch.Tensor):
        parameters = [parameters]

    if optimizer == 'adam':
        optimizer = optim.Adam(lr=lr, params=parameters)
    elif optimizer == 'sgd':
        optimizer = optim.SGD(lr=lr, params=parameters)
    elif optimizer == 'nesterov':
        optimizer = optim.SGD(lr=lr, params=parameters, nesterov=True, momentum=momentum, weight_decay=decay)
    else:
        raise NotImplementedError
    return optimizer

def init_loss(loss):
    if loss == 'cross_entropy':
        loss_fn = nn.CrossEntropyLoss()
    elif loss == 'mse':
        loss_fn = nn.MSELoss()
    elif loss == 'l1':
        loss_fn = nn.L1Loss()
    else:
        raise NotImplementedError
    return loss_fn

def gradient_matching(trigger_grads, poison_grads):
    """
    Compute the blind passenger loss term.
    """
    matching = 0
    poison_norm = 0
    source_norm = 0

    for poison_grad, trigger_grad in zip(poison_grads, trigger_grads):
        matching += (trigger_grad * poison_grad).sum()
        poison_norm += poison_grad.pow(2).sum()
        source_norm += trigger_grad.pow(2).sum()

    matching = matching / poison_norm.sqrt() / source_norm.sqrt()
    return 1 - matching

class VideoWriter:
    def __init__(self, out_path, codec='mp4v', fps=1, display=True):
        Path(out_path).parent.mkdir(exist_ok=True, parents=True)
        self.out_path = out_path
        self.video_writer = None
        self.fps = fps
        self.codec = codec
        self.display = display

    def write(self, im: torch.Tensor):
        if self.video_writer is None:
            shape = tuple(im.shape[1:3])
            codec = cv2.VideoWriter_fourcc(*self.codec)
            self.video_writer = cv2.VideoWriter(self.out_path, codec, self.fps, shape)

        im = im.clip(0, 1)
        im = im.permute(1, 2, 0)
        im = im.detach().numpy()
        im = (im * 255).astype(np.uint8)

        if self.display:
            try:
                plt.imshow(im)
                plt.pause(0.001)
            except AttributeError:
                print('\rCannot show figure', end='')

        im = cv2.cvtColor(im, cv2.COLOR_RGB2BGR)
        self.video_writer.write(im)

    def release(self):
        self.video_writer.release()

def log_to_file(log_path, config_dict, mean_results):
    with open(log_path, 'a') as f:
        output = dict(config_dict)
        output.update(mean_results)
        f.write(f'{output}\n')

class Mixup(torch.nn.Module):
    """This is data augmentation via mixup."""

    def __init__(self, nway=2, alpha=1.0):
        """Implement differentiable mixup, mixing nway-many examples with the given mixing factor alpha."""
        super().__init__()
        self.nway = nway
        self.mixing_alpha = alpha

    def forward(self, x, y, epoch=None):
        if self.mixing_alpha > 0:
            lmb = np.random.dirichlet([self.mixing_alpha] * self.nway, size=1).tolist()[0]
            batch_size = x.shape[0]
            indices = [torch.randperm(batch_size, device=x.device) for _ in range(self.nway)]
            mixed_x = sum(l * x[index, :] for l, index in zip(lmb, indices))
            y_s = [y[index] for index in indices]
        else:
            mixed_x = x
            y_s = y
            lmb = 1

        return mixed_x, y_s, lmb

    def corrected_loss(self, outputs, extra_labels, lmb=(1.0,), loss_fn=torch.nn.CrossEntropyLoss()):
        """Compute the corrected loss under consideration of the mixing."""
        predictions = torch.argmax(outputs.data, dim=1)
        correct_preds = sum([w * predictions.eq(l.data).cpu().numpy().mean() for w, l in zip(lmb, extra_labels)])
        loss = sum(weight * loss_fn(outputs, label) for weight, label in zip(lmb, extra_labels))
        return loss, correct_preds


## Dataset Classes

Custom PyTorch `Dataset` classes for poisoned data, triggered data,
and the two trigger-injection mechanisms (additive perturbation and
adaptive patch).

Originally from: **`utils/datasets.py`**


In [6]:
# ── Originally from: utils/datasets.py ─────────────────────────────


class PoisonedSubset(Dataset):
    def __init__(self, clean_dataset, indexes, norm, eps, clip=(0., 1.)):
        self.clean_dataset = clean_dataset
        self.indexes = indexes
        self.poison = torch.zeros((len(indexes), *clean_dataset[0][0].shape), requires_grad=True)

        self.clip = clip
        self.norm = norm
        self.eps = eps

    def __getitem__(self, item):
        x, y = self.clean_dataset[self.indexes[item]]
        x = x + self.poison[item]
        if self.clip is not None:
            if isinstance(x, torch.Tensor):
                x = x.clip(*self.clip)
            elif isinstance(x, (np.ndarray, np.generic)):
                x = np.clip(x, *self.clip)
            else:
                raise NotImplementedError
        return x, y

    def __len__(self):
        return len(self.indexes)

    def normalize_poison(self):
        poison = self.poison.detach()

        if self.norm == 'l_inf':
            poison = poison.clip(-self.eps, self.eps)
        elif self.norm == 'l2':
            norm = poison.norm(2, dim=(1, 2, 3), keepdim=True)
            poison[norm.flatten() > self.eps] = poison[norm.flatten() > self.eps] / norm[
                norm.flatten() > self.eps] * self.eps

        # self.poison = Settings.Transform.transform_tensor(self.poison)
        poison.requires_grad_()
        self.poison = poison


class PoisonedDataset(Dataset):
    """
    This is the poisoned dataset containing the whole clean data with the poisoned data
    """

    def __init__(self, clean_dataset, eps, norm, indexes, enabled_indexes=None):
        """
        enabled indexes - for evaluating filtering defences
        """
        self.clean_dataset = clean_dataset
        self.indexes = indexes
        self.poison_subset = PoisonedSubset(clean_dataset=clean_dataset, indexes=self.indexes, norm=norm, eps=eps)

        self.eps = eps
        self.norm = norm

        self.enabled_indexes = enabled_indexes

    def __getitem__(self, item):
        if self.enabled_indexes is not None:
            item = self.enabled_indexes[item]

        if item in self.indexes:
            i = self.indexes.index(item)
            return self.poison_subset[i]
        return self.clean_dataset[item]

    def __len__(self):
        if self.enabled_indexes is not None:
            return len(self.enabled_indexes)
        return len(self.clean_dataset)

    def normalize_poison(self):
        self.poison_subset.normalize_poison()


class TriggeredDataset(Dataset):
    """
    This dataset contains data with the trigger.
    It filters out the target labeled samples and keep only the samples with different class.
    """

    def __init__(self, clean_dataset, source_label, target_label, trigger_fn):
        self.clean_dataset = clean_dataset
        self.target_label = target_label
        self.source_label = source_label
        self.trigger_fn = trigger_fn

        self.labels = {i: x[1] for i, x in enumerate(clean_dataset)}
        self.source_label_samples = dict(filter(lambda x: x[1] == source_label, self.labels.items()))
        self.samples_list = list(self.source_label_samples.keys())

        self.label_to_return = 'both'  # options: 'both', 'true', 'target'
        print('Trigger dataset returns both true label and target label. '
              'To change it, change the `label_to_return` variable.')

        print('Creating triggered dataset:')
        print(f'source label: {self.clean_dataset.classes[source_label]} ({source_label}), '
              f'target label: {self.clean_dataset.classes[target_label]} ({target_label})')

    def __getitem__(self, item):
        x, y = self.clean_dataset[self.samples_list[item]]
        x = self.trigger_fn(x)
        if self.label_to_return == 'both':
            return x, y, self.target_label
        elif self.label_to_return == 'true':
            return x, y
        elif self.label_to_return == 'target':
            return x, self.target_label

    def __len__(self):
        return len(self.samples_list)


class TriggerAdditive:
    def __init__(self, size, method='randn', sigma=0.1, clip_lim=(0, 1), clip_trigger=16/255, **kwargs):
        self.clip_lim = clip_lim
        self.clip_trigger = clip_trigger

        if method == 'randn':
            self.trigger = torch.randn(size=size) * sigma
        else:
            raise NotImplementedError

        self.trigger.requires_grad_()

    def inject(self, sample):
        sample = sample + self.trigger.clip(-self.clip_trigger, self.clip_trigger)
        sample = torch.clip(sample, self.clip_lim[0], self.clip_lim[1])
        return sample

    def update_trigger(self, trigger):
        self.trigger = trigger
        self.trigger.requires_grad_()


class TriggerAdaptivePatch:
    def __init__(self, method='randn', sigma=0.01, clip_lim=(0, 1), patch_size=(3, 11, 11),
                 patch_path=None, trigger_loc='rand', trigger=None, **kwargs):
        """
        :param trigger_loc: `rand` or tuple of location (y, x)
        """
        self.clip_lim = clip_lim
        self.patch_size = patch_size
        self.trigger_loc = trigger_loc
        self.patch_size = patch_size

        if method == 'randn':
            self.trigger = torch.randn(size=patch_size) * sigma + 0.5
        elif method == 'from_file' and patch_path is not None:
            im_bgr = cv2.imread(patch_path)
            im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
            im = cv2.resize(im_rgb, patch_size[1:3]) / 255
            self.trigger = torch.tensor(im.transpose(2, 0, 1))
        elif method == 'from_tensor':
            if trigger is None:
                raise FileNotFoundError('if `from_tensor` is chosen, must give an argument `trigger`')
            self.trigger = trigger
        else:
            raise NotImplementedError

        self.trigger.requires_grad_()

    def update_trigger(self, trigger):
        self.trigger = trigger
        self.trigger.requires_grad_()

    def inject(self, sample):
        if (np.array(sample.shape) - np.array(self.patch_size)).min() < 0:
            raise ValueError('the patch size cannot be larger than the image')
        if self.trigger_loc == 'rand':
            _, y, x = sample.shape
            x = np.random.randint(0, x - self.patch_size[2] + 1)
            y = np.random.randint(0, y - self.patch_size[1] + 1)
        else:
            y, x = self.trigger_loc

        sample[:, y:y + self.patch_size[1], x:x + self.patch_size[2]] = self.trigger
        sample = torch.clip(sample, self.clip_lim[0], self.clip_lim[1])
        return sample


## Model Definitions

ResNet-18, VGG-11, and MobileNet-V2 adapted for CIFAR-10 (10 classes).
These are the surrogate / victim model architectures used in the attack.

Originally from: **`utils/utils.py`**


In [7]:
# ── Originally from: utils/utils.py ────────────────────────────────


def resnet18(pretrained=False, out_features=10):
    if pretrained:
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    else:
        model = models.resnet18()
    return nn.Sequential(*list(model.children())[:-1], torch.nn.Flatten(), nn.Linear(512, out_features))


def vgg11(pretrained=False, out_features=10):
    if pretrained:
        model = models.vgg11(weights=models.VGG11_Weights.IMAGENET1K_V1)
    else:
        model = models.vgg11()
    head = list(model.children())[0]
    flatten = nn.Flatten()
    linear1 = nn.Linear(in_features=512, out_features=512)
    relu1 = nn.ReLU(inplace=True)
    do1 = nn.Dropout(p=0.5, inplace=False)
    linear2 = nn.Linear(512, 512)
    relu2 = nn.ReLU(inplace=True)
    do2 = nn.Dropout(p=0.5, inplace=False)
    output = nn.Linear(512, out_features)
    classifier = nn.Sequential(linear1, relu1, do1, linear2, relu2, do2, output)
    return nn.Sequential(head, flatten, classifier)


def mobilenet(pretrained=False, out_features=10):
    if pretrained:
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights)
    else:
        model = models.mobilenet_v2()
    head = list(model.children())[:-1]
    classifier = nn.Sequential(nn.Flatten(), nn.Dropout(p=0.2), nn.Linear(1280, out_features=out_features))
    model = nn.Sequential(*head, classifier)
    return model


def get_model(model):
    if model == 'resnet18':
        model = resnet18()
    elif model == 'vgg11':
        model = vgg11()
    elif model == 'mobilenet_v2':
        model = mobilenet()
    return model


## Trigger Optimization

Functions for optimising the trigger pattern *before* poison crafting.
For additive triggers, PGD-style optimisation is used.
For adaptive-patch triggers, Adam-based optimisation is used.

Originally from: **`utils/poison_optimizer.py`**


In [8]:
# ── Originally from: utils/poison_optimizer.py ─────────────────────


def optimize_poison(model, dataset, normalizer, trigger, source_label, target_label, patch_size, trigger_loc,
                    device='cuda', batch_size=256, lr=0.01, epochs=100, scheduler_step=200, scheduler_gamma=0.1,
                    verbose=1, **kwargs):
    if isinstance(device, str):
        device = torch.device(device)
    model.eval()
    model.to(device)
    loss_fn = torch.nn.CrossEntropyLoss()

    # prepare dataset
    source_indexes = get_label_indexes(dataset=dataset, label=source_label)
    target = torch.ones(batch_size, dtype=torch.long) * target_label
    target = target.to(device)

    injector = TriggerAdaptivePatch('from_tensor', trigger=trigger, patch_size=patch_size, trigger_loc=trigger_loc)

    optimizer = torch.optim.Adam(params=[injector.trigger], lr=lr)
    scheduler = StepLR(optimizer, step_size=scheduler_step, gamma=scheduler_gamma)

    history = []
    for i in range(epochs):
        optimizer.zero_grad()

        triggered = sample_batch(batch_size, dataset, injector, source_indexes)
        loss, pred = optimization_step(model, triggered, normalizer, target, optimizer, scheduler, loss_fn, device)

        mean_sm = pred.softmax(dim=1).mean(dim=0).detach().cpu().numpy()
        loss_i = loss.detach().cpu().tolist()
        history.append((loss_i, mean_sm[source_label], mean_sm[target_label]))
        if verbose > 0:
            print(f'\r[{i + 1}] loss: {loss_i:.4f} | SM source {mean_sm[source_label]:.4f} | '
                  f'SM target {mean_sm[target_label]:.4f}', end='')
    return injector.trigger.detach()


def optimize_poison_additive(model, dataset, normalizer, device, source_label, target_label, eps, lr=1 / 255,
                             batch_size=256, epochs=500, **kwargs):
    # Create initial poison
    im, _ = dataset[0]
    noise = torch.zeros_like(im, requires_grad=True)

    # find source indexes and make the target label
    source_indexes = [i for i, (_, t) in enumerate(dataset) if t == source_label]
    target = torch.ones(batch_size, dtype=torch.long) * target_label

    loss_fn = torch.nn.CrossEntropyLoss()
    model.eval()
    for i in range(epochs):
        # make poisoned input samples
        inds = np.random.choice(source_indexes, size=batch_size, replace=False)
        triggered = [dataset[ind][0] + noise for ind in inds]
        triggered = torch.stack(triggered)

        # training step
        pred = model(normalizer(triggered).to(device))
        loss = loss_fn(pred, target.to(device))
        grads = torch.autograd.grad(loss, noise)
        noise = noise - grads[0].sign() * lr
        noise = noise.detach().clip(-eps, eps)
        noise.requires_grad_()

        # history
        mean_sm = pred.softmax(dim=1).mean(dim=0).detach().cpu().numpy()
        loss_i = loss.detach().cpu().tolist()
        print(
            f'\r[{i + 1}] loss: {loss_i:.4f} | SM source {mean_sm[source_label]:.4f} | SM target {mean_sm[target_label]:.4f} | {noise.abs().max():.5f}',
            end='')
    return noise.detach()


def optimization_step(model, batch, normalizer, target, optimizer, scheduler, loss_fn, device):
    pred = model(normalizer(batch).to(device))
    loss = loss_fn(pred, target)
    loss.backward()
    optimizer.step()
    scheduler.step()
    return loss, pred


def sample_batch(batch_size, dataset, injector, source_indexes):
    inds = np.random.choice(source_indexes, size=batch_size, replace=False)
    triggered = [injector.inject(dataset[ind][0]) for ind in inds]
    triggered = torch.stack(triggered)
    return triggered


def get_label_indexes(dataset, label):
    labels = np.array(list(map(lambda x: x[1], dataset)))
    indexes = np.where(labels == label)[0]
    return indexes


## Poison Crafting (Silent Killer)

The core of the Silent Killer attack: `PoisonCrafter` performs
gradient-alignment-based poison crafting.  It trains a surrogate model,
selects which training samples to poison, and iteratively refines
perturbations so that victim models trained on the poisoned dataset
learn the backdoor.

Originally from: **`utils/crafter.py`**


In [9]:
# ── Originally from: utils/crafter.py ──────────────────────────────


def get_trigger_function(trigger_type, **kwargs):
    if trigger_type == 'additive':
        return TriggerAdditive(**kwargs)
    elif trigger_type == 'adaptive_patch':
        return TriggerAdaptivePatch(**kwargs)


class PoisonCrafter:
    def __init__(self, model_initializer, clean_dataset, test_dataset, source_label, target_label, n_samples,
                 victim_lr, victim_momentum, victim_weight_decay, victim_milestones, victim_gamma, victim_loss,
                 victim_batch_size, victim_optimizer, alpha_poison, alpha_trigger, crafting_repetitions,
                 poison_selection, trigger_batch_size, trigger_init_method, log_wandb, trigger_type, device, patch_path,
                 trigger_loc, eps_p, eps_t, retraining_factor, retraining_epochs, retraining_batch_size,
                 retraining_loss, normalizer: transforms = transforms.Compose([]), log_freq=10, augmentations=True,
                 model_path=None, patch_size=8, train_print_freq=5, norm='l_inf'):
        self.device = device
        self.log_wandb = log_wandb

        self.r = None
        self.log_freq = log_freq
        self.n_samples = n_samples
        self.alpha_poison = alpha_poison
        self.alpha_trigger = alpha_trigger
        self.crafting_repetitions = crafting_repetitions
        self.poison_selection = poison_selection  # 'random' or 'gradient'
        self.patch_size = (3, patch_size, patch_size)
        self.trigger_batch_size = trigger_batch_size
        self.eps_p = eps_p
        self.eps_t = eps_t
        self.norm = norm

        # victim training parameters
        self.victim_lr = victim_lr
        self.victim_momentum = victim_momentum
        self.victim_weight_decay = victim_weight_decay
        self.victim_milestones = victim_milestones
        self.victim_gamma = victim_gamma
        self.victim_batch_size = victim_batch_size
        self.victim_optimizer = victim_optimizer
        self.train_print_freq = train_print_freq

        # retraining parameters
        self.retraining_factor = retraining_factor
        self.retraining_epochs = retraining_epochs
        self.retraining_batch_size = retraining_batch_size
        self.retraining_loss = retraining_loss

        # training settings
        if model_initializer == 'resnet18':
            self.model_initializer = resnet18
        elif model_initializer == 'vgg11':
            self.model_initializer = vgg11
        elif model_initializer == 'mobilenet_v2':
            self.model_initializer = mobilenet
        else:
            raise NotImplementedError
        self.loss_fn = init_loss(self.retraining_loss)
        self.victim_loss = init_loss(victim_loss)

        if augmentations:
            self.augmentations = Compose([RandomHorizontalFlip(p=0.5), RandomCrop(32, padding=4)])
        else:
            self.augmentations = transforms.Compose([])

        # data
        self.clean_dataset = clean_dataset
        self.test_dataset = test_dataset
        self.normalizer = normalizer

        # Create trigger set
        self.trigger_type = trigger_type
        self.source_class = source_label
        self.target_label = target_label
        self.patch_path = patch_path
        self.trigger_loc = trigger_loc

        self.trigger_injector = get_trigger_function(trigger_type, size=clean_dataset[0][0].shape, sigma=0.01,
                                                     patch_size=self.patch_size, method=trigger_init_method,
                                                     patch_path=patch_path, trigger_loc=trigger_loc)
        self.trigger_set = TriggeredDataset(clean_dataset=clean_dataset,
                                                     source_label=source_label, target_label=target_label,
                                                     trigger_fn=self.trigger_injector.inject)
        # create model
        self.model, self.optimizer, self.scheduler = self._init_model(lr=victim_lr, momentum=victim_momentum,
                                                                      weight_decay=victim_weight_decay,
                                                                      gamma=self.victim_gamma,
                                                                      milestones=self.victim_milestones)
        if model_path is not None:
            print('load model from file')
            self.model.load_state_dict(torch.load(model_path))
        else:
            print('train initial model from scratch')
            self._train(self.clean_dataset)

        # Create poison set
        indexes = self._choose_indexes()
        self.poisoned_dataset = PoisonedDataset(clean_dataset=clean_dataset, eps=self.eps_p,
                                                         norm=self.norm, indexes=indexes)

    def optimize_trigger(self, epochs, eps, lr, gamma):
        trigger = self.trigger_injector.trigger
        if self.trigger_type == 'adaptive_patch':
            opt_fn = optimize_poison
        elif self.trigger_type == 'additive':
            opt_fn = optimize_poison_additive
        else:
            raise NotImplementedError
        optimized_trigger = opt_fn(model=self.model, dataset=self.clean_dataset, normalizer=self.normalizer,
                                   device=self.device, trigger=trigger, source_label=self.source_class,
                                   trigger_loc=self.trigger_loc, epochs=epochs, eps=eps, lr=lr,
                                   target_label=self.target_label, patch_size=self.patch_size,
                                   trigger_type=self.trigger_type, scheduler_gamma=gamma)
        self.trigger_injector.update_trigger(optimized_trigger)

    def _choose_indexes(self):
        method = self.poison_selection
        if method == 'random':
            indexes = self._random_poison_selection()
        elif method == 'gradient':
            indexes = self._gradient_based_poison_selection(out_of_class=False, model=self.model)
        elif method == 'out_of_class_random':
            indexes = self._random_ooc_poison_selection()
        elif method == 'out_of_class_gradient':
            indexes = self._gradient_based_poison_selection(out_of_class=True, model=self.model)
        else:
            raise NotImplementedError
        print(f'Poisoning set, {len(indexes)}/{len(self.clean_dataset)} samples were randomly chose to poison, '
              f'{len(indexes) / len(self.clean_dataset) * 100:.2f}% of the dataset.')
        return indexes

    def _random_poison_selection(self):
        print('Random poison selection')
        indexes = [i for i, x in enumerate(self.clean_dataset) if x[1] == self.target_label]
        shuffle(indexes)
        indexes = indexes[:self.n_samples]
        return indexes

    def _random_ooc_poison_selection(self):
        """
        Choose poison samples of other classes than the target class
        """
        print('Random out-of-class poison selection')
        indexes = [i for i, x in enumerate(self.clean_dataset) if x[1] != self.target_label]
        shuffle(indexes)
        indexes = indexes[:self.n_samples]
        return indexes

    def _gradient_based_poison_selection(self, model, out_of_class=False):
        print('Gradient based poison selection')
        grad_dict = {}
        model.eval()
        parameters = [p for p in model.parameters() if p.requires_grad]
        for i, (x, y) in enumerate(DataLoader(self.clean_dataset, batch_size=1)):
            x, y = x.to(self.device), y.to(self.device)
            if i % 1000 == 0:
                print(f'\r[{i}/{len(self.clean_dataset)}] Choosing largest grad samples', end='')
            if (not out_of_class) and (y != self.target_label):
                continue
            elif out_of_class and (y == self.target_label):
                continue
            pred = model(self.normalizer(x))
            loss = self.victim_loss(pred, y)
            grads = torch.autograd.grad(loss, parameters)
            grads_norm = torch.stack(list(map(lambda p: p.norm(), grads))).sum()
            grad_dict[i] = grads_norm.tolist()
            if DEBUG:
                break
        grad_dict = dict(sorted(grad_dict.items(), key=lambda item: item[1], reverse=True))
        indexes = list(grad_dict.keys())[:self.n_samples]

        print('')
        return indexes

    def craft(self):
        r = self.crafting_repetitions  # R in the original paper
        t = self.retraining_factor  # T in the original paper
        for self.r in range(r):
            print(f'\nREPETITION {self.r + 1}:')
            if ((self.r + 1) % np.ceil(r / t) == 0) and ((self.r + 1) != r):
                self.model, self.optimizer, self.scheduler = self._init_model(lr=self.victim_lr,
                                                                              momentum=self.victim_momentum,
                                                                              weight_decay=self.victim_weight_decay,
                                                                              gamma=self.victim_gamma,
                                                                              milestones=self.victim_milestones)
                self._train(self.poisoned_dataset)

            self._optimize_poison()
            if DEBUG:
                break
        return self.poisoned_dataset

    def _optimize_poison(self):
        self.model.eval()

        batch_size = self.trigger_batch_size
        dataloader_train = DataLoader(self.trigger_set, batch_size=batch_size, shuffle=True)

        # training loop
        loss = 0
        for step, (x_batch, y_batch, y_target) in enumerate(dataloader_train):
            loss += self._compute_loss(x_batch, y_target)
            if DEBUG:
                break

        self._poison_optimization_step(loss)
        loss = loss.detach().cpu().tolist()
        print(f'\rPoison optimization | matching loss: {loss:.4f}')
        self.log_poisoning(loss=loss, log_wandb=self.log_wandb)

    def _poison_optimization_step(self, loss):
        grads = torch.autograd.grad(loss, (self.trigger_injector.trigger, self.poisoned_dataset.poison_subset.poison))

        trigger = self.trigger_injector.trigger.detach()
        trigger -= grads[0].sign() * self.alpha_trigger
        trigger = torch.clip(trigger, -self.eps_t, self.eps_t)
        trigger.requires_grad_()
        self.trigger_injector.trigger = trigger

        poison = self.poisoned_dataset.poison_subset.poison.detach()
        poison -= grads[1].sign() * self.alpha_poison
        poison = torch.clip(poison, -self.eps_p, self.eps_p)
        poison.requires_grad_()
        self.poisoned_dataset.poison_subset.poison = poison

    def _compute_loss(self, x_batch, y_target):
        poison_grads = self._compute_poison_gradient()
        trigger_grads = self._compute_trigger_gradient(x_batch, y_target)
        loss = gradient_matching(trigger_grads, poison_grads)
        return loss

    def log_poisoning(self, loss, log_wandb):
        im_p, _ = self.poisoned_dataset.poison_subset[0]
        im_c, _ = self.poisoned_dataset.poison_subset.clean_dataset[self.poisoned_dataset.poison_subset.indexes[0]]
        im_p, im_c = [im.cpu().detach().numpy().transpose((1, 2, 0)) for im in [im_p, im_c]]
        im_d = np.abs(im_p - im_c)

        im_t = [im for (im, _, _), _ in zip(self.trigger_set, range(3))]
        im_t = [im.cpu().detach().numpy().transpose((1, 2, 0)) for im in im_t]

        images = [PIL.Image.fromarray(np.uint8(image * 255)) for image in [im_p, im_c, im_d]]
        images_t = [PIL.Image.fromarray(np.uint8(image * 255)) for image in im_t]

        if log_wandb:
            wandb.log({"examples": [wandb.Image(image) for image in images],
                       'triggered_images': [wandb.Image(image) for image in images_t],
                       'max_diff': im_d.max(), 'diff_norm': np.linalg.norm(im_d), 'matching_loss': loss})

    def _compute_trigger_gradient(self, x, y):
        x, y = x.to(self.device), y.to(self.device)
        pred = self.model(self.normalizer(x))
        loss = self.victim_loss(pred, y)

        model_params = [p for p in self.model.parameters() if p.requires_grad]
        return torch.autograd.grad(loss, model_params, retain_graph=True, create_graph=True)

    def _compute_poison_gradient(self):
        loss = 0
        for data in DataLoader(self.poisoned_dataset.poison_subset, batch_size=self.victim_batch_size):
            x_poison, y_poison = list(map(lambda x: x.to(self.device), data))
            x_poison = self.augmentations(x_poison)
            pred = self.model(self.normalizer(x_poison))
            loss += self.victim_loss(pred, y_poison)
        loss /= len(self.poisoned_dataset)
        model_params = [p for p in self.model.parameters() if p.requires_grad]
        return torch.autograd.grad(loss, model_params, retain_graph=True, create_graph=True)

    def _init_model(self, lr, momentum, weight_decay, gamma, milestones):
        model = self.model_initializer().to(self.device)
        optimizer = init_optimizer(self.victim_optimizer, model.parameters(), lr,
                                   momentum=momentum, decay=weight_decay)
        scheduler = MultiStepLR(optimizer, gamma=gamma, milestones=milestones)
        return model, optimizer, scheduler

    def _train(self, dataset):
        for epoch in range(self.retraining_epochs):
            loss_list = []
            dataloader = DataLoader(dataset, batch_size=self.retraining_batch_size, shuffle=True)
            for i, (x_batch, y_batch) in enumerate(dataloader):
                x_batch, y_batch = x_batch.to(self.device), y_batch.to(self.device)
                self.model.train()
                loss = self._train_step(x_batch, y_batch, augmentation=self.augmentations)
                loss_list.append(loss.cpu().detach().tolist())

                if (i + 1) % self.log_freq == 0:
                    print(f'\rEpoch {epoch + 1} | step {i + 1}/{len(dataset) // self.retraining_batch_size} | '
                          f'loss: {np.array(loss_list).mean():.5f}', end='')
                if DEBUG:
                    break
            if DEBUG:
                break

            if (epoch + 1) % self.train_print_freq == 0 or epoch == self.retraining_epochs - 1:
                self.print_metrics(dataset)

            self.scheduler.step()

    def print_metrics(self, dataset):
        loss_train, acc_train = self._compute_acc(dataset)
        loss_test, acc_test = self._compute_acc(self.test_dataset)
        self.trigger_set.label_to_return = 'target'
        loss_trigger, acc_trigger = self._compute_acc(self.trigger_set)
        self.trigger_set.label_to_return = 'both'

        print(f' | train: loss {loss_train:.4f}, acc {acc_train:.4f}', end='')
        print(f' | test: loss {loss_test:.4f}, acc {acc_test:.4f}', end='')
        print(f' | trigger: loss {loss_trigger:.4f}, ASR {acc_trigger:.4f}')

        if self.log_wandb:
            wandb.log({'r': self.r, 'loss_train': loss_train, 'acc_train': acc_train,
                       'loss_test': loss_test, 'acc_test': acc_test,
                       'loss_trigger': loss_trigger, 'acc_trigger': acc_trigger})

    def _compute_acc(self, dataset):
        self.model.eval()
        acc_list = []
        loss_list = []
        for x_batch, y_batch in DataLoader(dataset, batch_size=self.retraining_batch_size):
            loss, acc = self._eval_step(x_batch, y_batch)
            loss_list.append(loss)
            acc_list.extend(acc)

        loss = np.array(loss_list).mean()
        acc = np.array(acc_list).mean()

        return loss, acc

    def _train_step(self, x_batch, y_batch, augmentation=None):
        if augmentation is not None:
            x_batch = augmentation(x_batch)

        self.optimizer.zero_grad()
        pred = self.model(self.normalizer(x_batch))
        loss = self.loss_fn(pred, y_batch)
        loss.backward()
        self.optimizer.step()
        return loss

    def _eval_step(self, x_batch, y_batch):
        x_batch, y_batch = x_batch.to(self.device), y_batch.to(self.device)
        self.optimizer.zero_grad()
        pred = self.model(self.normalizer(x_batch))
        loss = self.loss_fn(pred, y_batch)
        return loss.cpu().detach().tolist(), (pred.argmax(dim=-1) == y_batch).cpu().tolist()


## Training Functions

The `Trainer` class handles victim-model training, including optional
defence mechanisms (DP-SGD gradient clipping/noise, Mixup augmentation).

Originally from: **`utils/trainer.py`**


In [10]:
# ── Originally from: utils/trainer.py ──────────────────────────────


class Trainer:
    def __init__(self, model, device, normalizer, loss, optimizer, momentum, batch_size, weight_decay, epochs, lr,
                 milestones, gamma, validation_frequency, augmentations=True, defences=None):
        """
        :param model:       a pytorch model
        :param device:      torch.device
        :param normalizer:  torchvision transform
        """
        # training params
        self.loss = loss
        self.optimizer_name = optimizer
        self.momentum = momentum
        self.batch_size = batch_size
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.lr = lr
        self.milestones = milestones
        self.gamma = gamma
        self.validation_frequency = validation_frequency
        if augmentations:
            self.augmentations = Compose([RandomHorizontalFlip(p=0.5), RandomCrop(32, padding=4)])
        else:
            self.augmentations = None

        self.device = device
        self.model = model.to(self.device)
        self.best_model = model
        self.best_epoch = None

        self.loss_fn = init_loss(loss)
        self.dataset_train = None
        self.dataset_test = None
        self.normalizer = normalizer
        self.optimizer = None

        self.defences = defences
        if defences['mixup']:
            self.mixup = Mixup()

    def fit(self, dataset, dataset_test: dict = (), early_stop_dataset=None):
        self.dataset_train = dataset
        self.dataset_test = dataset_test
        best_loss = np.inf
        best_acc = 0

        self.optimizer = init_optimizer(self.optimizer_name, self.model.parameters(), self.lr,
                                        decay=self.weight_decay, momentum=self.momentum)
        scheduler = MultiStepLR(self.optimizer, milestones=self.milestones, gamma=self.gamma)

        for epoch in range(self.epochs):
            self.fit_epoch(epoch)
            scheduler.step()

            test_set_exists = self.dataset_test is not None
            eval_step = (epoch + 1) % self.validation_frequency == 0
            final_step = epoch == self.epochs - 1
            if test_set_exists and (eval_step or final_step):
                for i, (name, val_set) in enumerate(self.dataset_test.items()):
                    # eval dataset
                    acc, loss = self.eval(val_set, name=name)

                    # saving the best model
                    if early_stop_dataset == i and loss < best_loss:
                        self.best_model = deepcopy(self.model)
                        self.best_epoch = epoch
                        best_acc = acc
                        best_loss = loss
            print('')

        print('Training complete.', end=' ')
        if self.best_epoch is None:
            print('The final model is the last one.')
        else:
            print(f'The final model was from epoch #{self.best_epoch + 1}, '
                  f'with best loss of {best_loss:.4f} and best accuracy of {best_acc:.4f}')
        self.model = self.best_model

    def eval(self, validation_set, name='test'):
        self.model.eval()
        epoch_loss_list = []
        epoch_pred = []
        for x_test, y_test in DataLoader(validation_set, batch_size=self.batch_size):
            x_test, y_test = x_test.to(self.device), y_test.to(self.device)
            loss, pred = self.eval_step(x_test, y_test)
            epoch_loss_list.append(loss.cpu().detach().tolist())
            epoch_pred.append(pred)
        predictions = np.concatenate(list(map(lambda x: np.array(x[0]), epoch_pred)))
        ys = np.concatenate(list(map(lambda x: np.array(x[1]), epoch_pred)))
        acc = accuracy_score(ys, predictions)
        print(f' | {name} loss: {np.array(epoch_loss_list).mean():.4f} | {name} acc: {acc:.4f}', end='')
        return acc, np.array(epoch_loss_list).mean()

    def fit_epoch(self, epoch):
        self.model.train()
        epoch_loss_list = []
        epoch_pred = []
        for x_train, y_train in DataLoader(self.dataset_train, batch_size=self.batch_size, shuffle=True):
            self.fit_batch(epoch, epoch_loss_list, epoch_pred, x_train, y_train)
            if DEBUG:
                break
        predictions = np.concatenate(list(map(lambda x: np.array(x[0]), epoch_pred)))
        ys = np.concatenate(list(map(lambda x: np.array(x[1]), epoch_pred)))
        acc = accuracy_score(ys, predictions)
        print(f' | acc: {acc:.4f}', end='')

    def fit_batch(self, epoch, epoch_loss_list, epoch_pred, x_train, y_train):
        x_train, y_train = x_train.to(self.device), y_train.to(self.device)

        if self.augmentations is not None:
            x_train = self.augmentations(x_train)

        if self.defences['apply_defences'] and self.defences['mixup']:
            x_train, y_train, self.lmb = self.mixup(x_train, y_train)

        loss, pred = self.training_step(x_train, y_train)
        epoch_loss_list.append(loss.cpu().detach().tolist())
        epoch_pred.append(pred)
        print(f'\repoch [{epoch + 1}/{self.epochs}], loss: {np.array(epoch_loss_list).mean():.4f}', end='')

    def training_step(self, x_train, y_train):
        self.optimizer.zero_grad()
        pred = self.model(self.normalizer(x_train))

        if self.defences['apply_defences'] and self.defences['mixup']:
            loss, acc = self.mixup.corrected_loss(outputs=pred, extra_labels=y_train, lmb=self.lmb)
            y_train = y_train[0]
        else:
            loss = self.loss_fn(pred, y_train)
        loss.backward()

        # apply defences
        if self.defences['apply_defences']:
            self.apply_dpsgd()

        self.optimizer.step()
        return loss, (pred.argmax(dim=1).cpu().tolist(), y_train.cpu().tolist())

    def apply_dpsgd(self):
        apply_clip = 'dpsgd_clip' in self.defences.keys() and self.defences['dpsgd_clip'] is not None
        if apply_clip:
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.defences['dpsgd_clip'])
        if 'pdsgd_noise' in self.defences.keys():
            loc = torch.as_tensor(0., device=self.device)
            clip_factor = self.defences['dpsgd_clip'] if apply_clip else 1.0
            scale = torch.as_tensor(clip_factor * self.defences['pdsgd_noise'], device=self.device)
            generator = torch.distributions.normal.Normal(loc=loc, scale=scale)
            for param in self.model.parameters():
                param.grad += generator.sample(param.shape)

    def eval_step(self, x_test, y_test):
        pred = self.model(self.normalizer(x_test))
        loss = self.loss_fn(pred, y_test)
        return loss, (pred.argmax(dim=1).cpu().tolist(), y_test.cpu().tolist())


## Evaluation Functions

Functions for evaluating the backdoor attack: training a fresh victim
model on the poisoned dataset and measuring ASR (Attack Success Rate)
and clean accuracy.  Also includes activation-clustering-based defence
evaluation.

Originally from: **`utils/trainer.py`**


In [11]:
# ── Originally from: utils/trainer.py ──────────────────────────────


def evaluate_poisoning(dataset_poisoned, dataset_test, dataset_trigger, normalizer, loss, optimizer, momentum,
                       batch_size, weight_decay, epochs, lr, milestones, gamma, validation_frequency, log_wandb,
                       device, model='resnet18', defences=None):
    if defences['apply_defences'] and defences['activation_clustering']:
        activation_clustering(batch_size, dataset_poisoned, dataset_test, dataset_trigger, defences, epochs,
                              gamma, loss, lr, milestones, model, momentum, normalizer, optimizer,
                              validation_frequency, weight_decay, device=device)
    model = get_model(model)
    trainer = Trainer(model=model, device=device, normalizer=normalizer, loss=loss, optimizer=optimizer,
                      momentum=momentum, batch_size=batch_size, weight_decay=weight_decay, epochs=epochs, lr=lr,
                      milestones=milestones, gamma=gamma, validation_frequency=validation_frequency, defences=defences)
    trainer.fit(dataset_poisoned, {'clean_val': dataset_test, 'triggered_val': dataset_trigger}, early_stop_dataset=0)
    asr, trigger_set_loss = trainer.eval(dataset_trigger)
    clean_acc, clean_loss = trainer.eval(dataset_test)
    results = {'ASR': asr, 'Clean Acc': clean_acc,
               'clean loss': clean_loss, 'trigger set loss': trigger_set_loss}
    if log_wandb:
        wandb.log(results)
    return results


def activation_clustering(batch_size, dataset_poisoned, dataset_test, dataset_trigger, defences, epochs, gamma, loss,
                          lr, milestones, model, momentum, normalizer, optimizer, validation_frequency, weight_decay,
                          device):
    print('train model for activation clustering')
    feature_extractor, model = train_feature_extractor(batch_size, dataset_poisoned, dataset_test, dataset_trigger,
                                                       defences, epochs, gamma, loss, lr, milestones, model,
                                                       momentum, normalizer, optimizer, validation_frequency,
                                                       weight_decay, device=device)
    print('extract features')
    class_indices, features = extract_features(dataset_poisoned, feature_extractor, device=device)
    print('find clean samples')
    clean_indices = filter_poison(class_indices, features)
    dataset_poisoned.enabled_indexes = clean_indices
    print(f'new size: {len(dataset_poisoned)}')


def train_feature_extractor(batch_size, dataset_poisoned, dataset_test, dataset_trigger, defences, epochs, gamma, loss,
                            lr, milestones, model, momentum, normalizer, optimizer, validation_frequency, weight_decay,
                            device):
    model = get_model(model)
    trainer = Trainer(model=model, device=device, normalizer=normalizer, loss=loss, optimizer=optimizer,
                      momentum=momentum, batch_size=batch_size, weight_decay=weight_decay, epochs=epochs, lr=lr,
                      milestones=milestones, gamma=gamma, validation_frequency=validation_frequency, defences=defences)
    trainer.fit(dataset_poisoned, {'clean_val': dataset_test, 'triggered_val': dataset_trigger}, early_stop_dataset=0)
    layer_cake = list(model.children())
    feature_extractor = torch.nn.Sequential(*(layer_cake[:-1]), torch.nn.Flatten())
    return feature_extractor, model


def filter_poison(class_indices, features):
    clean_indices = []
    for i in class_indices.keys():
        if len(class_indices[i]) > 1:
            temp_feats = np.array([features[temp_idx].squeeze(0).cpu().numpy() for temp_idx in class_indices[i]])
            kmeans = KMeans(n_clusters=2).fit(temp_feats)
            clean_label = np.median(kmeans.labels_)
            clean = [class_indices[i][idx] for idx, is_clean in enumerate(kmeans.labels_ == clean_label) if is_clean]
            clean_indices = clean_indices + clean
    return clean_indices


def extract_features(dataset_poisoned, feature_extractor, device):
    features = []
    class_indices = defaultdict(list)
    with torch.no_grad():
        for i, (img, source) in tqdm(enumerate(dataset_poisoned), total=len(dataset_poisoned)):
            img = img.unsqueeze(0).to(device)
            features.append(feature_extractor(img))
            class_indices[source].append(i)
    return class_indices, features


## Experiment Setup

Select the device, load CIFAR-10, derive `n_samples` from `POISON_RATE`,
create output directories, and build the configuration dict for logging.
Nothing here needs to be edited.


In [12]:
# -- Device ----------------------------------------------------
device = get_device(DEVICE_NAME)

# -- Create output directories ---------------------------------
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# -- Load CIFAR-10 ---------------------------------------------
dataset_train = datasets.CIFAR10(
    root=DATA_ROOT, train=True,  download=True, transform=transforms.ToTensor())
dataset_test  = datasets.CIFAR10(
    root=DATA_ROOT, train=False, download=True, transform=transforms.ToTensor())
normalizer = get_normalizer(dataset_train)

# -- Derive n_samples from POISON_RATE -------------------------
# This is the ONLY place n_samples is defined in the notebook.
n_samples = max(1, int(POISON_RATE * len(dataset_train)))
print(f"Poison rate {POISON_RATE:.1%} -> n_samples = {n_samples} "
      f"(out of {len(dataset_train)} training samples)")

# -- Defences dict ----------------------------------------------
defences = {
    'apply_defences':        APPLY_DEFENCES,
    'dpsgd_clip':            DPSGD_CLIP,
    'dpsgd_noise':           DPSGD_NOISE,
    'activation_clustering': ACTIVATION_CLUSTERING,
    'mixup':                 MIXUP,
}

# -- Configuration dict (for W&B / log_to_file) -----------------
config_dict = dict(
    seed=SEED,
    source_label=SOURCE_LABEL,    target_label=TARGET_LABEL,
    poison_rate=POISON_RATE,      n_samples=n_samples,
    crafting_repetitions=CRAFTING_REPETITIONS,
    alpha_poison=ALPHA_POISON,    alpha_trigger=ALPHA_TRIGGER,
    poison_selection=POISON_SELECTION,
    eps=EPS,                      perturbations_norm=PERTURBATIONS_NORM,
    retraining_factor=RETRAINING_FACTOR,
    retraining_batch_size=RETRAINING_BATCH_SIZE,
    retraining_epochs=RETRAINING_EPOCHS,
    retraining_loss=RETRAINING_LOSS,
    trigger_type=TRIGGER_TYPE,    trigger_opt=TRIGGER_OPT,
    patch_path=PATCH_PATH,        trigger_loc=str(TRIGGER_LOC),
    trigger_init_method=TRIGGER_INIT_METHOD,
    trigger_batch_size=TRIGGER_BATCH_SIZE,
    patch_size=PATCH_SIZE,
    trigger_opt_epochs=TRIGGER_OPT_EPOCHS,
    trigger_opt_eps=TRIGGER_OPT_EPS,
    trigger_opt_lr=TRIGGER_OPT_LR,
    trigger_opt_gamma=TRIGGER_OPT_GAMMA,
    victim_milestones=str(VICTIM_MILESTONES),
    victim_gamma=VICTIM_GAMMA,    victim_lr=VICTIM_LR,
    victim_momentum=VICTIM_MOMENTUM,
    victim_weight_decay=VICTIM_WEIGHT_DECAY,
    victim_loss=VICTIM_LOSS,      victim_optimizer=VICTIM_OPTIMIZER,
    victim_batch_size=VICTIM_BATCH_SIZE,
    victim_epochs=VICTIM_EPOCHS,
    validation_frequency=VALIDATION_FREQUENCY,
    victim_augmentations=VICTIM_AUGMENTATIONS,
    craft_model=CRAFT_MODEL,      eval_model=EVAL_MODEL,
    val_repetitions=VAL_REPETITIONS,
    apply_defences=APPLY_DEFENCES,
    activation_clustering=ACTIVATION_CLUSTERING,
    mixup=MIXUP,                  dpsgd_clip=str(DPSGD_CLIP),
    dpsgd_noise=DPSGD_NOISE,
    train_print_freq=TRAIN_PRINT_FREQ,
    device=DEVICE_NAME,           use_wandb=USE_WANDB,
    wandb_entity=str(WANDB_ENTITY), project=PROJECT, run_name=RUN_NAME,
    log_path=str(LOG_PATH),
)

# -- Optional W&B init -----------------------------------------
if USE_WANDB:
    init_wandb(entity=WANDB_ENTITY, project=PROJECT, name=RUN_NAME,
               config_dict=config_dict)

print("Experiment setup complete.")


using CUDA


100%|██████████| 170M/170M [1:38:49<00:00, 28.8kB/s]


Normalization computed mean: tensor([0.4869, 0.4771, 0.4391]), std: tensor([0.2443, 0.2402, 0.2525])
Poison rate 1.0% -> n_samples = 500 (out of 50000 training samples)
Experiment setup complete.


## Experiment Summary

A concise overview of the experiment that is about to run.
Review this before committing to a long training session.


In [13]:
_W = 64
_hr = '=' * _W
_mr = '-' * _W
_cls = dataset_train.classes

print(_hr)
print('  SILENT KILLER - EXPERIMENT CONFIGURATION')
print(_hr)
print(f'  Seed               : {SEED}')
print(f'  Device             : {DEVICE_NAME}')
print(f'  Dataset            : CIFAR-10  (train={len(dataset_train)}, test={len(dataset_test)})')
print(f'  Source class        : {SOURCE_LABEL}  ({_cls[SOURCE_LABEL]})')
print(f'  Target class        : {TARGET_LABEL}  ({_cls[TARGET_LABEL]})')
print(f'  Poison rate         : {POISON_RATE:.1%}  ->  n_samples = {n_samples}')
print(f'  Trigger type        : {TRIGGER_TYPE}')
print(f'  Trigger epsilon     : {EPS:.6f}  ({EPS * 255:.1f}/255)')
print(f'  Trigger optimisation: {"enabled (" + str(TRIGGER_OPT_EPOCHS) + " epochs)" if TRIGGER_OPT else "disabled"}')
print(f'  Craft model         : {CRAFT_MODEL}')
print(f'  Eval model          : {EVAL_MODEL}')
print(f'  Crafting reps (R)   : {CRAFTING_REPETITIONS}')
print(f'  Victim epochs       : {VICTIM_EPOCHS}')
print(f'  Victim batch size   : {VICTIM_BATCH_SIZE}')
print(f'  Poison selection    : {POISON_SELECTION}')
print(_mr)
print('  PIPELINE FLAGS')
print(_mr)
print(f'  RUN_CLEAN_MODEL     : {RUN_CLEAN_MODEL}')
print(f'  RUN_POISONED_MODEL  : {RUN_POISONED_MODEL}')
print(f'  RUN_EVALUATION      : {RUN_EVALUATION}')
print(f'  REUSE_CHECKPOINTS   : {REUSE_CHECKPOINTS}')
print(_mr)
print('  CHECKPOINTS')
print(_mr)
_ce = Path(CLEAN_CKPT_PATH).exists()
_pe = Path(POISONED_CKPT_PATH).exists()
print(f'  Clean checkpoint    : {CLEAN_CKPT_PATH}')
print(f'    on disk           : {"YES" if _ce else "NO  (will be created)"}')
print(f'  Poisoned checkpoint : {POISONED_CKPT_PATH}')
print(f'    on disk           : {"YES" if _pe else "NO  (will be created)"}')
print(_hr)


  SILENT KILLER - EXPERIMENT CONFIGURATION
  Seed               : 2027
  Device             : cuda
  Dataset            : CIFAR-10  (train=50000, test=10000)
  Source class        : 3  (cat)
  Target class        : 0  (airplane)
  Poison rate         : 1.0%  ->  n_samples = 500
  Trigger type        : additive
  Trigger epsilon     : 0.062745  (16.0/255)
  Trigger optimisation: enabled (500 epochs)
  Craft model         : resnet18
  Eval model          : resnet18
  Crafting reps (R)   : 250
  Victim epochs       : 80
  Victim batch size   : 128
  Poison selection    : gradient
----------------------------------------------------------------
  PIPELINE FLAGS
----------------------------------------------------------------
  RUN_CLEAN_MODEL     : True
  RUN_POISONED_MODEL  : True
  RUN_EVALUATION      : True
  REUSE_CHECKPOINTS   : True
----------------------------------------------------------------
  CHECKPOINTS
----------------------------------------------------------------
  Clean c

## Clean Model: Load or Train

**Decision logic (evaluated in priority order):**

1. `REUSE_CHECKPOINTS=True` **and** checkpoint exists -> load it, skip training.
2. `RUN_CLEAN_MODEL=True` -> train from scratch and save with metadata.
3. Neither -> raise `RuntimeError` (no silent failures).

The clean surrogate is the mandatory starting point for poison crafting.
Each checkpoint stores both the model weights **and** experiment metadata.


In [14]:
# -- Internal weights-only path ---------------------------------
# PoisonCrafter expects a raw state_dict for model_path.
# We store the full checkpoint (weights + metadata) in CLEAN_CKPT_PATH
# and a weights-only copy for the crafter.
_WEIGHTS_PATH = str(Path(CHECKPOINT_DIR) / '_clean_weights_only.pt')


def _make_clean_meta():
    """Build experiment metadata stored alongside the clean checkpoint."""
    return {
        'checkpoint_type': 'clean_surrogate',
        'seed':            SEED,
        'source_label':    SOURCE_LABEL,
        'target_label':    TARGET_LABEL,
        'poison_rate':     POISON_RATE,
        'trigger_type':    TRIGGER_TYPE,
        'trigger_epsilon': EPS,
        'craft_model':     CRAFT_MODEL,
        'eval_model':      EVAL_MODEL,
        'epochs':          RETRAINING_EPOCHS,
        'timestamp':       datetime.datetime.now().isoformat(),
    }


# -- Branch: load or train --------------------------------------
if REUSE_CHECKPOINTS and Path(CLEAN_CKPT_PATH).exists():
    # Path 1: load existing checkpoint
    print(f'[CHECKPOINT] Loading clean model from: {CLEAN_CKPT_PATH}')
    _ckpt        = torch.load(CLEAN_CKPT_PATH, map_location=device)
    _clean_state = _ckpt['model_state_dict']
    _clean_meta  = _ckpt.get('metadata', {})
    print(f'[CHECKPOINT] Metadata: {_clean_meta}')
    torch.save(_clean_state, _WEIGHTS_PATH)
    print('[CHECKPOINT] Clean model loaded. Training skipped.')

elif RUN_CLEAN_MODEL:
    # Path 2: train clean surrogate from scratch
    print('[TRAINING] Training clean surrogate model from scratch ...')
    _clean_model = get_model(CRAFT_MODEL).to(device)
    _clean_opt   = init_optimizer(
        VICTIM_OPTIMIZER, _clean_model.parameters(), VICTIM_LR,
        momentum=VICTIM_MOMENTUM, decay=VICTIM_WEIGHT_DECAY)
    _clean_sched = MultiStepLR(
        _clean_opt, gamma=VICTIM_GAMMA, milestones=list(VICTIM_MILESTONES))
    _clean_aug   = (Compose([RandomHorizontalFlip(p=0.5), RandomCrop(32, padding=4)])
                    if VICTIM_AUGMENTATIONS else transforms.Compose([]))
    _clean_loss  = init_loss(RETRAINING_LOSS)

    for _epoch in range(RETRAINING_EPOCHS):
        _loss_list = []
        _clean_model.train()
        _loader = DataLoader(
            dataset_train, batch_size=RETRAINING_BATCH_SIZE, shuffle=True)
        for _i, (_xb, _yb) in enumerate(_loader):
            _xb, _yb = _xb.to(device), _yb.to(device)
            _xb = _clean_aug(_xb)
            _clean_opt.zero_grad()
            _pred = _clean_model(normalizer(_xb))
            _loss = _clean_loss(_pred, _yb)
            _loss.backward()
            _clean_opt.step()
            _loss_list.append(_loss.cpu().detach().item())
            if (_i + 1) % TRAIN_PRINT_FREQ == 0:
                print(f'\rEpoch [{_epoch + 1}/{RETRAINING_EPOCHS}] '
                      f'step {_i + 1} | loss: {np.mean(_loss_list):.5f}', end='')
            if DEBUG:
                break
        _clean_sched.step()
        if DEBUG:
            break
    print()

    _clean_state = _clean_model.state_dict()
    _clean_meta  = _make_clean_meta()

    # Save full checkpoint (weights + metadata)
    torch.save({'model_state_dict': _clean_state, 'metadata': _clean_meta},
               CLEAN_CKPT_PATH)
    # Save weights-only for PoisonCrafter
    torch.save(_clean_state, _WEIGHTS_PATH)
    print(f'[CHECKPOINT] Clean surrogate saved to: {CLEAN_CKPT_PATH}')

else:
    # Path 3: no checkpoint, no training -> fail loudly
    raise RuntimeError(
        f'No clean model checkpoint found at "{CLEAN_CKPT_PATH}" '
        f'and RUN_CLEAN_MODEL=False. '
        f'Either set REUSE_CHECKPOINTS=True with an existing checkpoint, '
        f'or set RUN_CLEAN_MODEL=True to train from scratch.'
    )

print("Clean model ready.")


[TRAINING] Training clean surrogate model from scratch ...
Epoch [40/40] step 390 | loss: 0.32752
[CHECKPOINT] Clean surrogate saved to: ./checkpoints/clean_surrogate.pt
Clean model ready.


## Poison Crafting (Silent Killer)

Initialise `PoisonCrafter` from the **clean surrogate**, optionally
optimise the trigger pattern, and run the gradient-alignment crafting loop.

> **The Silent Killer algorithm is preserved verbatim.** This cell only
> manages configuration wiring, checkpoint saving, and pipeline flags.


In [ ]:
if RUN_POISONED_MODEL:
    # -- Initialise PoisonCrafter from the clean surrogate ------
    print('[CRAFTING] Initialising PoisonCrafter from clean surrogate ...')
    poison_crafter = PoisonCrafter(
        model_initializer=CRAFT_MODEL,
        model_path=_WEIGHTS_PATH,
        clean_dataset=dataset_train,
        test_dataset=dataset_test,
        normalizer=normalizer,
        augmentations=VICTIM_AUGMENTATIONS,
        source_label=SOURCE_LABEL,
        target_label=TARGET_LABEL,
        n_samples=n_samples,
        victim_loss=VICTIM_LOSS,
        victim_optimizer=VICTIM_OPTIMIZER,
        victim_batch_size=VICTIM_BATCH_SIZE,
        victim_lr=VICTIM_LR,
        victim_momentum=VICTIM_MOMENTUM,
        victim_weight_decay=VICTIM_WEIGHT_DECAY,
        victim_milestones=VICTIM_MILESTONES,
        poison_selection=POISON_SELECTION,
        victim_gamma=VICTIM_GAMMA,
        crafting_repetitions=CRAFTING_REPETITIONS,
        alpha_poison=ALPHA_POISON,
        alpha_trigger=ALPHA_TRIGGER,
        trigger_batch_size=TRIGGER_BATCH_SIZE,
        patch_size=PATCH_SIZE,
        trigger_init_method=TRIGGER_INIT_METHOD,
        log_wandb=USE_WANDB,
        trigger_type=TRIGGER_TYPE,
        device=device,
        eps_p=EPS,
        eps_t=TRIGGER_OPT_EPS,
        patch_path=PATCH_PATH,
        trigger_loc=TRIGGER_LOC,
        train_print_freq=TRAIN_PRINT_FREQ,
        norm=PERTURBATIONS_NORM,
        retraining_factor=RETRAINING_FACTOR,
        retraining_loss=RETRAINING_LOSS,
        retraining_batch_size=RETRAINING_BATCH_SIZE,
        retraining_epochs=RETRAINING_EPOCHS,
    )

    # -- Trigger optimisation (optional) -----------------------
    if TRIGGER_OPT:
        print('[TRIGGER] Optimising trigger pattern ...')
        poison_crafter.optimize_trigger(
            epochs=TRIGGER_OPT_EPOCHS,
            eps=TRIGGER_OPT_EPS,
            lr=TRIGGER_OPT_LR,
            gamma=TRIGGER_OPT_GAMMA,
        )

    # -- Run gradient-alignment crafting loop -------------------
    print('[CRAFTING] Running Silent Killer gradient-alignment loop ...')
    poisoned_dataset = poison_crafter.craft()

    # -- Save poisoned surrogate checkpoint (weights + metadata)
    _poison_meta = {
        'checkpoint_type':       'poisoned_surrogate',
        'seed':                  SEED,
        'source_label':          SOURCE_LABEL,
        'target_label':          TARGET_LABEL,
        'poison_rate':           POISON_RATE,
        'n_samples':             n_samples,
        'trigger_type':          TRIGGER_TYPE,
        'trigger_epsilon':       EPS,
        'craft_model':           CRAFT_MODEL,
        'eval_model':            EVAL_MODEL,
        'epochs':                VICTIM_EPOCHS,
        'crafting_repetitions':  CRAFTING_REPETITIONS,
        'timestamp':             datetime.datetime.now().isoformat(),
    }
    torch.save(
        {'model_state_dict': poison_crafter.model.state_dict(),
         'metadata': _poison_meta},
        POISONED_CKPT_PATH,
    )
    print(f'[CHECKPOINT] Poisoned surrogate saved to: {POISONED_CKPT_PATH}')

    # -- Build test trigger set for evaluation -----------------
    test_trigger_set = TriggeredDataset(
        clean_dataset=dataset_test,
        source_label=SOURCE_LABEL,
        target_label=TARGET_LABEL,
        trigger_fn=poison_crafter.trigger_set.trigger_fn,
    )
    test_trigger_set.label_to_return = 'target'

    print('Poison crafting complete.')

else:
    print('[SKIP] RUN_POISONED_MODEL=False - skipping poison crafting.')


## Evaluation

Train a **fresh victim model** on the poisoned dataset and measure:
- **Clean Accuracy** on the unmodified test set.
- **Attack Success Rate (ASR)** on triggered source-class samples.


In [ ]:
if RUN_EVALUATION:
    # Guard: crafting must have run first
    if 'poisoned_dataset' not in dir() or 'test_trigger_set' not in dir():
        raise RuntimeError(
            'Cannot evaluate: poisoned_dataset or test_trigger_set is not defined. '
            'Run the Poison Crafting cell first with RUN_POISONED_MODEL=True.'
        )

    print('[EVALUATION] Training fresh victim model on poisoned dataset ...')
    _n = VAL_REPETITIONS if not DEBUG else 1
    results_df = pd.DataFrame([
        evaluate_poisoning(
            poisoned_dataset, dataset_test, test_trigger_set, normalizer,
            milestones=VICTIM_MILESTONES,
            gamma=VICTIM_GAMMA,
            model=EVAL_MODEL,
            loss=VICTIM_LOSS,
            optimizer=VICTIM_OPTIMIZER,
            momentum=VICTIM_MOMENTUM,
            batch_size=VICTIM_BATCH_SIZE,
            weight_decay=VICTIM_WEIGHT_DECAY,
            epochs=VICTIM_EPOCHS,
            lr=VICTIM_LR,
            device=device,
            validation_frequency=VALIDATION_FREQUENCY,
            defences=defences,
            log_wandb=USE_WANDB,
        )
        for _ in range(_n)
    ])
    mean_results = {f'mean {k}': v for k, v in dict(results_df.mean()).items()}

    if USE_WANDB:
        wandb.log({'results': wandb.Table(dataframe=results_df), **mean_results})

    if LOG_PATH is not None:
        log_to_file(LOG_PATH, config_dict, mean_results)

    print('\n' + '=' * 64)
    print('  EXPERIMENT COMPLETE')
    print('=' * 64)

else:
    print('[SKIP] RUN_EVALUATION=False - skipping evaluation.')


## Results: Clean Accuracy & Attack Success Rate (ASR)

Display the evaluation metrics from the experiment.


In [17]:
if 'results_df' in dir() and results_df is not None:
    print('=' * 64)
    print('         SILENT KILLER - EXPERIMENT RESULTS')
    print('=' * 64)
    print(f'  Source label  : {SOURCE_LABEL} ({dataset_test.classes[SOURCE_LABEL]})')
    print(f'  Target label  : {TARGET_LABEL} ({dataset_test.classes[TARGET_LABEL]})')
    print(f'  Trigger type  : {TRIGGER_TYPE}')
    print(f'  Craft model   : {CRAFT_MODEL}')
    print(f'  Eval model    : {EVAL_MODEL}')
    print(f'  Poison rate   : {POISON_RATE:.1%}  (n_samples = {n_samples})')
    print(f'  Epsilon       : {EPS:.6f}')
    print('-' * 64)
    for col in results_df.columns:
        vals = results_df[col].values
        print(f'  {col:22s}: {vals.mean():.4f}  '
              f'(per-run: {[round(v, 4) for v in vals.tolist()]})')
    print('-' * 64)
    for k, v in mean_results.items():
        print(f'  {k:24s}: {v:.4f}')
    print('=' * 64)
else:
    print('No results available. Run evaluation first (RUN_EVALUATION=True).')


         SILENT KILLER - EXPERIMENT RESULTS
  Source label  : 3 (cat)
  Target label  : 0 (airplane)
  Trigger type  : additive
  Craft model   : resnet18
  Eval model    : resnet18
  Poison rate   : 1.0%  (n_samples = 500)
  Epsilon       : 0.062745
----------------------------------------------------------------
  ASR                   : 0.9780  (per-run: [0.978])
  Clean Acc             : 0.8374  (per-run: [0.8374])
  clean loss            : 0.4885  (per-run: [0.4885])
  trigger set loss      : 0.0866  (per-run: [0.0866])
----------------------------------------------------------------
  mean ASR                : 0.9780
  mean Clean Acc          : 0.8374
  mean clean loss         : 0.4885
  mean trigger set loss   : 0.0866


## Trigger Visualization

Visualise the learned trigger pattern and show how it modifies
source-class samples.


In [ ]:
if 'poison_crafter' in dir():
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    # 1) Trigger pattern (normalised for display)
    trigger_tensor = poison_crafter.trigger_injector.trigger.detach().cpu()
    trigger_np = trigger_tensor.numpy().transpose(1, 2, 0)
    trigger_disp = (trigger_np - trigger_np.min()) / (trigger_np.max() - trigger_np.min() + 1e-8)
    axes[0].imshow(trigger_disp)
    axes[0].set_title('Trigger (normalised)')
    axes[0].axis('off')

    # 2) Clean source-class sample
    src_idx = [i for i, (_, y) in enumerate(dataset_test) if y == SOURCE_LABEL][0]
    clean_img, _ = dataset_test[src_idx]
    axes[1].imshow(clean_img.numpy().transpose(1, 2, 0))
    axes[1].set_title(f'Clean ({dataset_test.classes[SOURCE_LABEL]})')
    axes[1].axis('off')

    # 3) Triggered sample
    triggered_img = poison_crafter.trigger_injector.inject(clean_img.clone())
    axes[2].imshow(triggered_img.detach().numpy().transpose(1, 2, 0))
    axes[2].set_title('With trigger')
    axes[2].axis('off')

    # 4) Difference (amplified for visibility)
    diff = np.abs(triggered_img.detach().numpy() - clean_img.numpy()).transpose(1, 2, 0)
    diff_amp = diff / (diff.max() + 1e-8)
    axes[3].imshow(diff_amp)
    axes[3].set_title('|Difference| (amplified)')
    axes[3].axis('off')

    plt.suptitle('Trigger Visualization', fontsize=14, fontweight='bold')
    plt.tight_layout()
    _trig_path = str(Path(RESULTS_DIR) / 'trigger_visualization.png')
    plt.savefig(_trig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {_trig_path}')
else:
    print('[SKIP] No poison_crafter found. '
          'Run poison crafting first (RUN_POISONED_MODEL=True).')


## Poisoned Sample Visualization

Compare original (clean) training samples with their poisoned versions.


In [ ]:
if 'poisoned_dataset' in dir():
    n_show = min(5, len(poisoned_dataset.indexes))
    fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 9))

    for i in range(n_show):
        idx = poisoned_dataset.indexes[i]
        clean_img, label = dataset_train[idx]
        poisoned_img, _ = poisoned_dataset.poison_subset[i]

        # Clean
        axes[0, i].imshow(clean_img.numpy().transpose(1, 2, 0))
        axes[0, i].set_title(f'Clean #{idx}\n({dataset_train.classes[label]})')
        axes[0, i].axis('off')

        # Poisoned
        axes[1, i].imshow(poisoned_img.detach().numpy().transpose(1, 2, 0))
        axes[1, i].set_title(f'Poisoned #{idx}')
        axes[1, i].axis('off')

        # Difference (amplified)
        diff = np.abs(poisoned_img.detach().numpy() - clean_img.numpy()).transpose(1, 2, 0)
        diff_amp = diff / (diff.max() + 1e-8)
        axes[2, i].imshow(diff_amp)
        axes[2, i].set_title('|Diff| (amplified)')
        axes[2, i].axis('off')

    axes[0, 0].set_ylabel('Clean', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Poisoned', fontsize=12, fontweight='bold')
    axes[2, 0].set_ylabel('Difference', fontsize=12, fontweight='bold')

    plt.suptitle('Clean vs Poisoned Samples', fontsize=14, fontweight='bold')
    plt.tight_layout()
    _ps_path = str(Path(RESULTS_DIR) / 'poisoned_samples.png')
    plt.savefig(_ps_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {_ps_path}')
else:
    print('[SKIP] No poisoned_dataset found. '
          'Run poison crafting first (RUN_POISONED_MODEL=True).')


## Save Experimental Results

Save evaluation results to CSV and configuration + mean results to JSON.


In [20]:
if 'results_df' in dir() and results_df is not None:
    import json as _json

    # Results DataFrame -> CSV
    _csv_path = str(Path(RESULTS_DIR) / 'experiment_results.csv')
    results_df.to_csv(_csv_path, index=False)
    print(f'Results saved to : {_csv_path}')

    # Config + mean results -> JSON
    _config_out = {k: str(v) for k, v in config_dict.items()}
    _config_out.update({k: str(v) for k, v in mean_results.items()})
    _json_path = str(Path(RESULTS_DIR) / 'experiment_config.json')
    with open(_json_path, 'w') as _f:
        _json.dump(_config_out, _f, indent=2)
    print(f'Config saved to  : {_json_path}')

    print('\n' + '=' * 64)
    print('  All outputs saved. Experiment finished successfully!')
    print('=' * 64)
else:
    print('No results to save. Run evaluation first (RUN_EVALUATION=True).')


Results saved to : results/experiment_results.csv
Config saved to  : results/experiment_config.json

  All outputs saved. Experiment finished successfully!
